In [ ]:
import os
import gc
import gc
import glob
import numpy as np
import nibabel as nib
from PIL import Image

In [ ]:
INPUT_DIR = "C:/users/madhu/IDP/image_processing/datasets/PKG - UTSW-Glioma"
OUTPUT_DIR = "C:/users/madhu/IDP/image_processing/datasets/utsw_slices"

MODALITIES = {
    "t1" : "*T1*.nii.gz",
    "t1c" : "*T1c*.nii.gz",
    "t2" : "*T2*.nii.gz",
    "flair":"*FLAIR*.nii.gz"
}

SEG_PATTERN = "*seg*.nii.gz"

AXIS = 2 # slicing axis info: 0=sagittal, 1=coronal, 2=axial
OUTPUT_FORMAT ="png"
NORMALIZE = True

#N4 bias field correction
APPLY_N4 = True
N4_SHRINK = 4
N4_ITERATIONS - [50, 50, 50]

#Intensity normalization
NORM_LOW_PCT = 1.0
NORM_HIGH_PCT = 99.0

#clahe application
APPLY_CLAHE = True
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID = (8, 8)
SAVE_NO_CLAHE_COPY = False

#slice filtering
SKIP_EMPTY_SEG_SLICES = False  #keeping slices where the mask has a labeled tumor  
MIN_NONZERO_FRACTION = 0.0 

In [ ]:
def inspect_structure(root=INPUT_DIR, max_patients=2, max_files=15):
    patient_dirs = sorted(
        d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d)
    )
    print(f"{len(patient_dirs)} patient folders found under {root}\n")
    for pdir in patient_dirs[:max_patients]:
        print(f"--- {os.path.basename(pdir)} ---")
        files = glob.glob(os.path.join(pdir, "**", "*"), recursive=True)
        for f in files[:max_files]:
            print("  ", os.path.relpath(f, pdir))
        print()

In [ ]:
def find_patient_dirs(root):
    return sorted(d for d in glob.glob(os.path.join(root,"*")) if os.path.isdir(d))

In [ ]:
def find_one(patient_dir,pattern):
    matches = glob.glob(os.path.join(patient_dir,"**",pattern),recursive=True)
    return matches[0] if matches else None

In [ ]:
def already_done(patient_id):
    pdir = os.path.join(OUTPUT_DIR,patient_id)
    if not os.path.isdir(pdir):
        return False
    for _,_,files in os.walk(pdir):
        if files:
            return True
    return False

In [ ]:
def get_slice(dataobj,index,axis):
    """slicing a single 2d slice without getting the full 3d volume
    for space constraints"""
    idx = [slice(None)]*dataobj.ndim
    idx[axis] = index
    return np.asarray(dataobj[tuple(idx)])
    

In [ ]:
def normalize_slices(sl):
    sl = sl.astype(np.float32)
    lo, hi = np.percentile(sl,1), np.percentile(sl,99)
    if hi <= lo:
        return np.zeros_like(sl,dtype=np.uint8)
    sl = np.clip((sl - lo) / (hi - lo), 0, 1) #min-max scaling (value - min)/(value - max) 
    return (sl*255).astype(np.uint8)

In [ ]:
def save_slice(arr2d, out_path, is_mask = False):
    os.makedirs(os.path.dirname(out_path),exist_ok = True)
    if OUTPUT_FORMAT == "npy":
        np.save(out_path.replace(".png",".npy"),arr2d)
    else:
        img = (arr2d*255).astype(np.uint8) if is_mask else (
            normalize_slices(arr2d) if NORMALIZE else arr2d.astype(np.uint8)
        )
        Image.fromarray(img).save(out_path)

#the gm values in the sample data are probability values between 0.0 and 1.0
#if casted into uint8 without scaling it gets truncated the the dataset that we will be using will not have an issue so mul with 255 is just an adjustment for this


In [ ]:
def stream_patient(patient_dir, patient_id):
    """processing one patient's volume slice-by-slice, writing each slice to
    disk and the the already stored patient's data is deleted (for space constraints)"""
    seg_path = find_one(patient_dir, SEG_PATTERN)
    seg_img = nib.load(seg_path) if seg_path else None
    for mod_name, pattern in MODALITIES.items():
        mod_path = find_one(patient_dir,pattern)
        if mod_path is None:
            continue
        img = nib.load(mod_path)
        n_slices = img.shape[AXIS]
        for i in range(n_slices):
            sl = get_slice(img.dataobj, i, AXIS)
            seg_slice = None
            if seg_img is not None:
                seg_slice = get_slice(seg_img.dataobj, i, AXIS)
            if SKIP_EMPTY_SEG_SLICES and seg_slice is not None and not np.any(seg_slice):
                del sl, seg_slice
                continue
            if MIN_NONZERO_FRACTION > 0 and (np.count_nonzero(sl) / sl.size) < MIN_NONZERO_FRACTION:
                del sl
                if seg_slice is not None:
                    del seg_slice
                continue
            out_img = os.path.join(OUTPUT_DIR, patient_id, mod_name, f"slice_{i:03d}.{OUTPUT_FORMAT}")
            save_slice(sl, out_img,is_mask=False)
            if seg_slice is not None:
                out_mask = os.path.join(OUTPUT_DIR,patient_id,"seg",f"slice_{i:03d}.{OUTPUT_FORMAT}")
                save_slice(seg_slice, out_mask, is_mask=True)
            del sl
            if seg_slice is not None:
                del seg_slice
        del img
        gc.collect()
    if seg_img is not None:
        del seg_img
    gc.collect()
    

In [ ]:
def main():
    os.makedirs(OUTPUT_DIR,exist_ok = True)
    patient_dirs = find_patient_dirs(INPUT_DIR)
    print(f"Found {len(patient_dirs)} patient folders under {INPUT_DIR}")
    for idx, pdir in enumerate(patient_dirs, 1):
        patient_id = os.path.basename(pdir)
        print(f"[{idx}/{len(patient_dirs)}] {patient_id}")
        try:
            stream_patient(pdir,patient_id)
        except Exception as e:
            print(f"patient skipped : {e}")
        gc.collect()
    print("Done. Slices written to",OUTPUT_DIR)

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
if __name__ == "__main__":
    inspect_structure(); exit()

In [ ]:
t1_dir = "/kaggle/working/slices/patient_000/t1"
print(f"{len(os.listdir(t1_dir))} slices generated")

In [ ]:
#looking at a slice
import matplotlib.pyplot as plt
from PIL import Image

mid = len(os.listdir(t1_dir)) // 2
img = Image.open(os.path.join(t1_dir,f"slice_{mid:03d}.png"))
plt.imshow(img,cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
#multiple slices at once
fig, axes = plt.subplots(1,6,figsize=(18,3))
n = len(os.listdir(t1_dir))
step = n // 6
for i, ax in enumerate(axes):
    idx = i*step
    img = Image.open(os.path.join(t1_dir,f"slice_{idx:03d}.png"))
    ax.imshow(img,cmap="gray")
    ax.set_title(f"slice {idx}")
    ax.axis("off")
plt.suptitle("T1 volume - evenly spaced slices")
plt.tight_layout()
plt.show()

In [ ]:
seg_dir = "/kaggle/working/slices/patient_000/seg"
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(Image.open(os.path.join(t1_dir, f"slice_{mid:03d}.png")), cmap="gray")
axes[0].set_title("T1")
axes[1].imshow(Image.open(os.path.join(seg_dir, f"slice_{mid:03d}.png")), cmap="gray")
axes[1].set_title("Segmentation mask") #this the ground truth maps the tumor or which pixels belong to a tumor
for a in axes: a.axis("off")
plt.show()